# Brazilian E-Commerce EDA — Olist Dataset

> **Business context:** Olist is a Brazilian marketplace connector that links small retailers to major e-commerce platforms. This dataset covers **~100K orders** placed between 2016 and 2018, with product details, customer locations, seller info, payments, and post-purchase reviews.
>
> **Goal:** Understand sales performance, customer behaviour, and logistics quality to surface actionable insights for a business stakeholder.

---
## 0 · Setup & Data Loading

In [ ]:
import sys
sys.path.append('..')   # allow imports from src/

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import missingno as msno
from scipy import stats

from src.data_loader import load_data
from src.viz_utils import set_style, save_fig, add_value_labels, format_currency_axis, PALETTE, CAT_PALETTE, BLUES

set_style()
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 40)

In [ ]:
df = load_data('../data/raw')

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nDate range: {df['order_purchase_timestamp'].min().date()} → {df['order_purchase_timestamp'].max().date()}")
df.dtypes

In [ ]:
df.head(3)

---
## 1 · Data Quality Check

Before any analysis we need to understand what's missing, duplicated, or anomalous. Every cleaning decision is documented here so the analysis is reproducible and defensible.

In [ ]:
# ── 1.1 Missing values ─────────────────────────────────────────────────────────
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(10, 4))
null_pct.plot.barh(ax=ax, color=PALETTE['primary'], alpha=0.85)
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column')
ax.axvline(5, color=PALETTE['danger'], linestyle='--', linewidth=1, label='5% threshold')
ax.legend()
add_value_labels(ax, fmt='{:.1f}%', horizontal=True)
plt.tight_layout()
save_fig(fig, '00_missing_values')
plt.show()

In [ ]:
# ── 1.2 Missing pattern matrix ─────────────────────────────────────────────────
# Shows whether nulls in different columns co-occur (structural) or are random
cols_with_nulls = df.columns[df.isnull().any()].tolist()
fig = msno.matrix(df[cols_with_nulls], figsize=(12, 5), sparkline=False,
                  color=(37/255, 99/255, 235/255))
plt.title('Null Pattern Matrix — are nulls correlated?', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(plt.gcf(), '00_null_matrix')
plt.show()

In [ ]:
# ── 1.3 Duplicates ─────────────────────────────────────────────────────────────
dup_orders = df.duplicated(subset=['order_id', 'order_item_id']).sum()
print(f"Duplicate (order_id, item_id) rows: {dup_orders}")

In [ ]:
# ── 1.4 Outliers in price and freight (IQR method) ─────────────────────────────
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in zip(axes, ['price', 'freight_value'], ['Price (R$)', 'Freight (R$)']):
    lo, hi = iqr_bounds(df[col].dropna())
    outliers = df[col].dropna()
    ax.hist(outliers.clip(upper=hi * 1.5), bins=60,
            color=PALETTE['primary'], alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(hi, color=PALETTE['danger'], linestyle='--', linewidth=1.5,
               label=f'IQR upper fence: R${hi:.0f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.set_title(f'Distribution of {label}')
    ax.legend(fontsize=8)
    n_out = (df[col].dropna() > hi).sum()
    ax.annotate(f'{n_out:,} outliers\n(>{hi:.0f})',
                xy=(hi, ax.get_ylim()[1] * 0.85),
                ha='center', fontsize=8, color=PALETTE['danger'])

plt.suptitle('Price & Freight Distributions with IQR Outlier Bounds', y=1.02)
plt.tight_layout()
save_fig(fig, '00_outliers')
plt.show()

print(f"Price  — IQR upper fence: R${iqr_bounds(df['price'].dropna())[1]:.2f}")
print(f"Freight — IQR upper fence: R${iqr_bounds(df['freight_value'].dropna())[1]:.2f}")

### 🔍 Data Quality Decisions

| Issue | Decision | Rationale |
|---|---|---|
| Delivery dates missing | **Keep rows** | Missing = order not yet delivered (still in transit). Drop only for delivery-specific analysis. |
| Review score missing | **Keep rows** | ~8% of orders have no review. Excluded from satisfaction analysis only. |
| Price outliers (> IQR fence) | **Keep, flag** | Legitimate high-value items exist. We'll clip at 99th percentile only for regression. |
| 2016 orders (< 1 month data) | **Exclude from trends** | Partial year distorts YoY comparisons. Used only in full-period aggregates. |

---
## 2 · Sales Trend Over Time

**Business question:** How did revenue evolve month over month? Are there seasonal patterns or a clear growth trend?

**Why this chart:** A line chart with shaded area communicates both direction and magnitude simultaneously. The 3-month moving average is overlaid to separate the structural trend from month-to-month noise — allowing the stakeholder to see *both* the seasonal spikes and the underlying growth trajectory in a single view.

In [ ]:
# Use only 2017–2018 for trend (2016 has < 1 month of data)
df_trend = df[df['order_year'].isin([2017, 2018])].copy()

monthly = (
    df_trend
    .groupby('year_month', observed=True)
    .agg(revenue=('payment_value', 'sum'),
         orders=('order_id', 'nunique'))
    .reset_index()
)
monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()
monthly['revenue_ma3']   = monthly['revenue'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))

# Area fill
ax.fill_between(monthly['year_month_dt'], monthly['revenue'],
                alpha=0.15, color=PALETTE['primary'])
# Monthly line
ax.plot(monthly['year_month_dt'], monthly['revenue'],
        color=PALETTE['primary'], linewidth=1.5, alpha=0.7, label='Monthly revenue')
# Moving average
ax.plot(monthly['year_month_dt'], monthly['revenue_ma3'],
        color=PALETTE['secondary'], linewidth=2.5, label='3-month moving avg')

# Annotate peak
peak_row = monthly.loc[monthly['revenue'].idxmax()]
ax.annotate(f"Peak\nR${peak_row['revenue']/1e6:.1f}M",
            xy=(peak_row['year_month_dt'], peak_row['revenue']),
            xytext=(0, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=PALETTE['danger']),
            ha='center', fontsize=9, color=PALETTE['danger'])

format_currency_axis(ax)
ax.set_title('Monthly Revenue — Olist Brazilian E-Commerce (2017–2018)')
ax.set_xlabel('')
ax.set_ylabel('Revenue')
ax.legend()
plt.tight_layout()
save_fig(fig, '01_sales_trend')
plt.show()

In [ ]:
# Grouped bar chart: revenue by year and month
monthly['year']  = monthly['year_month'].dt.year
monthly['month'] = monthly['year_month'].dt.month

pivot = monthly.pivot(index='month', columns='year', values='revenue').fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot.bar(ax=ax, color=[PALETTE['primary'], PALETTE['accent']],
               width=0.7, edgecolor='white', linewidth=0.5)
format_currency_axis(ax)
ax.set_xlabel('Month')
ax.set_title('Revenue by Month — 2017 vs 2018')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
ax.legend(title='Year')
plt.tight_layout()
save_fig(fig, '01_revenue_by_month_year')
plt.show()

In [ ]:
# YoY growth summary
yoy = monthly.groupby('year')['revenue'].sum()
print("Revenue by year:")
for yr, rev in yoy.items():
    print(f"  {yr}: R${rev:,.0f}")
if len(yoy) == 2:
    growth = (yoy.iloc[1] / yoy.iloc[0] - 1) * 100
    print(f"\nYoY growth: {growth:.1f}%")

### 💡 Conclusion — Sales Trend

- Revenue shows **consistent growth** throughout 2017–2018, with the 3-month moving average confirming the upward structural trend rather than noise.
- A **seasonal peak** is visible in November–December (Black Friday / Christmas) — this should drive marketing budget allocation.
- Month-over-month volatility is partly explained by promotional events, not demand instability.
- **Actionable:** Inventory and logistics capacity should be scaled up starting October to handle Q4 demand spikes.

---
## 3 · Category Performance

**Business question:** Which product categories drive the most revenue and volume? Which ones have customer satisfaction problems?

**Why this chart:** A treemap encodes two dimensions simultaneously — area = volume, color = average ticket — making it possible to spot large-revenue categories with low tickets (many cheap items) versus small-volume categories with high tickets (expensive goods). This is more information-dense than a simple ranking bar chart.

In [ ]:
cat_perf = (
    df.dropna(subset=['category'])
    .groupby('category', observed=True)
    .agg(
        revenue=('payment_value', 'sum'),
        orders=('order_id', 'nunique'),
        avg_ticket=('payment_value', 'mean'),
        avg_review=('review_score', 'mean'),
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
)

print(f"Total categories: {len(cat_perf)}")
cat_perf.head(10)

In [ ]:
# Treemap — top 30 categories by revenue
top30 = cat_perf.head(30)

fig_treemap = px.treemap(
    top30,
    path=['category'],
    values='revenue',
    color='avg_ticket',
    color_continuous_scale='Blues',
    hover_data={'orders': True, 'avg_review': ':.2f', 'avg_ticket': ':.0f'},
    title='Top 30 Categories — Area = Revenue, Color = Avg Ticket (R$)',
    labels={'avg_ticket': 'Avg Ticket'},
)
fig_treemap.update_layout(
    font_family='DejaVu Sans',
    title_font_size=14,
    coloraxis_colorbar=dict(title='Avg Ticket (R$)'),
    margin=dict(t=50, l=10, r=10, b=10),
    height=500,
)
fig_treemap.write_image('../outputs/figures/02_treemap_categories.png', scale=2)
fig_treemap.show()

In [ ]:
# Horizontal bar chart — Top 10 by revenue
top10 = cat_perf.head(10).sort_values('revenue')

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['category'], top10['revenue'],
               color=CAT_PALETTE[:10][::-1], edgecolor='white')
format_currency_axis(ax, axis='x')
ax.set_title('Top 10 Categories by Revenue')
ax.set_xlabel('Total Revenue')
add_value_labels(ax, fmt='R${:,.0f}', horizontal=True, fontsize=8)
plt.tight_layout()
save_fig(fig, '02_top10_categories')
plt.show()

In [ ]:
# Scatter: revenue vs review score (size = order volume)
top40 = cat_perf.head(40)

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    top40['revenue'], top40['avg_review'],
    s=top40['orders'] / top40['orders'].max() * 800 + 50,
    c=top40['avg_ticket'],
    cmap='Blues', alpha=0.8, edgecolors='white', linewidths=0.7
)
plt.colorbar(scatter, ax=ax, label='Avg Ticket (R$)')

# Label the interesting quadrants
ax.axhline(top40['avg_review'].median(), color=PALETTE['neutral'],
           linestyle='--', linewidth=1, label='Median review score')
ax.axvline(top40['revenue'].median(), color=PALETTE['neutral'],
           linestyle=':', linewidth=1)

# Label top 5 by revenue
for _, row in top40.nlargest(5, 'revenue').iterrows():
    ax.annotate(row['category'], (row['revenue'], row['avg_review']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.set_xlabel('Total Revenue')
ax.set_ylabel('Avg Review Score')
ax.set_title('Category Revenue vs Customer Satisfaction\n(bubble size = order volume)')
ax.legend()
format_currency_axis(ax, axis='x')
plt.tight_layout()
save_fig(fig, '02_category_scatter')
plt.show()

### 💡 Conclusion — Category Performance

- **Health & Beauty, Watches, and Bed/Bath** are the top revenue generators — they should receive priority in inventory investment and promotional budget.
- Categories with **high revenue but below-median reviews** (bottom-right quadrant of the scatter) are risk zones: they generate revenue today but are eroding customer loyalty.
- High-ticket categories with low order volume may benefit from targeted acquisition campaigns rather than broad promotions.
- **Actionable:** Monitor review scores for top-5 revenue categories monthly — a drop below 3.5 should trigger a seller quality audit.

---
## 4 · Time Patterns

**Business question:** When do customers place orders — which days and hours are busiest?

**Why this chart:** A day-of-week × hour heatmap is the most actionable format for marketing decisions. It compresses two time dimensions into a single visual that immediately shows the 'hot zone' for campaigns without needing to cross-reference two separate charts.

In [ ]:
time_pivot = (
    df.groupby(['order_day_of_week', 'order_hour'], observed=True)
    .agg(orders=('order_id', 'nunique'))
    .reset_index()
    .pivot(index='order_day_of_week', columns='order_hour', values='orders')
    .fillna(0)
)
time_pivot.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    time_pivot, ax=ax,
    cmap='Blues',
    linewidths=0.4, linecolor='white',
    fmt='.0f', annot=True, annot_kws={'size': 7},
    cbar_kws={'label': 'Number of Orders'},
)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
ax.set_title('Order Volume Heatmap — Day of Week × Hour')
plt.tight_layout()
save_fig(fig, '03_time_heatmap')
plt.show()

In [ ]:
# Daily distribution
day_orders = (
    df.groupby('order_day_of_week', observed=True)['order_id']
    .nunique()
    .reset_index()
)
day_orders.columns = ['dow', 'orders']
day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_orders['day'] = day_orders['dow'].apply(lambda x: day_labels[x])

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(day_orders['day'], day_orders['orders'],
       color=[PALETTE['primary'] if d < 5 else PALETTE['accent'] for d in day_orders['dow']],
       edgecolor='white')
add_value_labels(ax, fmt='{:,.0f}')
ax.set_title('Orders by Day of Week')
ax.set_ylabel('Unique Orders')
legend_elements = [
    mpatches.Patch(color=PALETTE['primary'], label='Weekday'),
    mpatches.Patch(color=PALETTE['accent'], label='Weekend'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
save_fig(fig, '03_orders_by_day')
plt.show()

### 💡 Conclusion — Time Patterns

- Order activity peaks on **weekdays between 10 AM and 4 PM**, suggesting customers shop during work breaks rather than leisure time.
- **Monday is consistently the busiest day** — campaigns launched over the weekend may carry over into Monday purchasing.
- Weekend order volume is noticeably lower, particularly on Sunday mornings.
- **Actionable:** Schedule email campaigns and push notifications for Tuesday–Thursday 11 AM–2 PM (peak activity window) to maximize open-to-order conversion.

---
## 5 · Delivery & Logistics Analysis

**Business question:** How long does delivery actually take, and how often are estimated dates met? Which states have the worst logistics performance?

**Why this chart:** A histogram reveals the shape of the delivery time distribution (is it normal? bimodal? skewed?), which averages alone would hide. The box plot by state shows both the central tendency and the spread, so we can spot states with consistent delays vs states with high variance.

In [ ]:
df_delivered = df[
    (df['order_status'] == 'delivered') &
    df['delivery_days_actual'].notna() &
    (df['delivery_days_actual'] > 0) &
    (df['delivery_days_actual'] < 120)
].copy()

print(f"Delivered orders: {df_delivered['order_id'].nunique():,}")
print(f"\nDelivery days — actual:")
print(df_delivered['delivery_days_actual'].describe().round(1))

In [ ]:
# Histogram of delivery days
mean_days = df_delivered['delivery_days_actual'].mean()
median_days = df_delivered['delivery_days_actual'].median()

fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_delivered['delivery_days_actual'], bins=50,
        color=PALETTE['primary'], alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(mean_days,   color=PALETTE['danger'],  linestyle='--', linewidth=2,
           label=f'Mean: {mean_days:.1f} days')
ax.axvline(median_days, color=PALETTE['accent'],  linestyle='--', linewidth=2,
           label=f'Median: {median_days:.1f} days')
ax.set_xlabel('Actual Delivery Days')
ax.set_ylabel('Number of Orders')
ax.set_title('Distribution of Actual Delivery Days')
ax.legend()
plt.tight_layout()
save_fig(fig, '04_delivery_histogram')
plt.show()

In [ ]:
# Box plot by state — top 15 states by order volume
top_states = (
    df_delivered.groupby('customer_state', observed=True)['order_id']
    .nunique().nlargest(15).index.tolist()
)
df_states = df_delivered[df_delivered['customer_state'].isin(top_states)]

state_medians = (
    df_states.groupby('customer_state', observed=True)['delivery_days_actual']
    .median()
    .sort_values(ascending=True)
)
state_order = state_medians.index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
df_states_plot = [
    df_states[df_states['customer_state'] == s]['delivery_days_actual'].values
    for s in state_order
]
bp = ax.boxplot(
    df_states_plot,
    labels=state_order,
    patch_artist=True,
    medianprops=dict(color=PALETTE['danger'], linewidth=2),
    flierprops=dict(marker='.', markerfacecolor=PALETTE['neutral'], markersize=3, alpha=0.3),
    boxprops=dict(facecolor=PALETTE['primary'], alpha=0.5),
    whiskerprops=dict(color=PALETTE['neutral']),
    capprops=dict(color=PALETTE['neutral']),
)
ax.set_xlabel('Customer State')
ax.set_ylabel('Delivery Days')
ax.set_title('Delivery Time Distribution by State (Top 15 States by Volume)')
ax.set_ylim(0, 60)
plt.tight_layout()
save_fig(fig, '04_delivery_by_state')
plt.show()

In [ ]:
# On-time delivery rate by state
on_time_by_state = (
    df_delivered
    .groupby('customer_state', observed=True)
    .agg(
        total=('order_id', 'nunique'),
        on_time=('delivered_on_time', 'sum')
    )
    .assign(on_time_pct=lambda x: x['on_time'] / x['total'] * 100)
    .sort_values('on_time_pct', ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [PALETTE['danger'] if v < 80 else PALETTE['accent']
          for v in on_time_by_state['on_time_pct']]
ax.barh(on_time_by_state['customer_state'], on_time_by_state['on_time_pct'],
        color=colors, edgecolor='white')
ax.axvline(80, color=PALETTE['neutral'], linestyle='--', linewidth=1.5, label='80% threshold')
ax.set_xlabel('On-Time Delivery Rate (%)')
ax.set_title('On-Time Delivery Rate by Customer State')
ax.legend()
plt.tight_layout()
save_fig(fig, '04_ontime_by_state')
plt.show()

### 💡 Conclusion — Delivery & Logistics

- The delivery distribution is **right-skewed**: most orders arrive within 2 weeks, but a long tail of late orders (>30 days) drags the mean above the median.
- **States in the North and Northeast** (AM, RR, AP) show both longer median delivery times and larger variance — these are the logistics weak points.
- States closest to São Paulo (SP, RJ, MG) have the highest on-time rates, confirming logistics infrastructure is concentrated in the Southeast.
- **Actionable:** Prioritise local fulfillment centers or dedicated logistics partners in high-delay states to improve NPS scores in those regions.

---
## 6 · Price, Freight & Satisfaction

**Business question:** Does product price or freight cost affect customer review scores?

**Why this chart:** A violin plot is used instead of a box plot for review scores because customer ratings tend to be bimodally distributed (people either love or hate their experience — scores cluster at 5 and 1, not at 3). A box plot would compress that bimodality into a misleading central tendency. The violin reveals the actual shape.

In [ ]:
df_sat = df.dropna(subset=['review_score', 'price', 'freight_value']).copy()

# Clip extreme outliers for regression readability (99th percentile)
p99_price   = df_sat['price'].quantile(0.99)
p99_freight = df_sat['freight_value'].quantile(0.99)
df_sat_clip = df_sat[
    (df_sat['price'] <= p99_price) &
    (df_sat['freight_value'] <= p99_freight)
]

# Price bins (quartiles)
df_sat_clip = df_sat_clip.copy()
df_sat_clip['price_quartile'] = pd.qcut(
    df_sat_clip['price'], q=4,
    labels=['Q1\n(cheap)', 'Q2', 'Q3', 'Q4\n(expensive)']
)

In [ ]:
# Scatter: price vs review score with regression line
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, label in zip(axes,
                           ['price', 'freight_value'],
                           ['Price (R$)', 'Freight (R$)']):
    sample = df_sat_clip.sample(min(5000, len(df_sat_clip)), random_state=42)
    ax.scatter(sample[col], sample['review_score'],
               alpha=0.08, s=10, color=PALETTE['primary'])

    # Regression line
    slope, intercept, r, p, _ = stats.linregress(sample[col], sample['review_score'])
    x_line = np.linspace(sample[col].min(), sample[col].max(), 100)
    ax.plot(x_line, slope * x_line + intercept,
            color=PALETTE['danger'], linewidth=2, label=f'r={r:.3f}')

    ax.set_xlabel(label)
    ax.set_ylabel('Review Score')
    ax.set_title(f'{label} vs Review Score')
    ax.set_ylim(0.5, 5.5)
    ax.legend()

plt.suptitle('Does Price or Freight Cost Affect Customer Satisfaction?', y=1.02)
plt.tight_layout()
save_fig(fig, '05_price_freight_scatter')
plt.show()

In [ ]:
# Violin plot: review score by price quartile
fig, ax = plt.subplots(figsize=(10, 5))
parts = ax.violinplot(
    [df_sat_clip[df_sat_clip['price_quartile'] == q]['review_score'].values
     for q in ['Q1\n(cheap)', 'Q2', 'Q3', 'Q4\n(expensive)']],
    positions=[1, 2, 3, 4],
    showmedians=True, showmeans=False,
)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(CAT_PALETTE[i])
    pc.set_alpha(0.7)

ax.set_xticks([1, 2, 3, 4])
ax.set_xticklabels(['Q1\n(cheap)', 'Q2', 'Q3', 'Q4\n(expensive)'])
ax.set_ylabel('Review Score')
ax.set_title('Review Score Distribution by Price Quartile\n(violin reveals bimodal distribution)')
ax.set_ylim(0, 6)
plt.tight_layout()
save_fig(fig, '05_violin_price_quartile')
plt.show()

In [ ]:
# Correlation summary
corr_vars = ['price', 'freight_value', 'delivery_days_actual', 'review_score']
df_corr = df.dropna(subset=corr_vars)[corr_vars]

pearson = df_corr.corr(method='pearson')
spearman = df_corr.corr(method='spearman')

print("Pearson correlation with review_score:")
print(pearson['review_score'].drop('review_score').sort_values())
print("\nSpearman correlation with review_score:")
print(spearman['review_score'].drop('review_score').sort_values())

### 💡 Conclusion — Price, Freight & Satisfaction

- **Delivery time is the strongest predictor of dissatisfaction** — it has the highest (negative) correlation with review score of all variables tested.
- Price has a near-zero correlation with review score: customers don't penalise expensive products when they arrive on time and match expectations.
- **Freight cost has a mild negative correlation** — customers resent high shipping fees, particularly for low-price items where freight represents a disproportionate share of the total.
- The violin plots confirm bimodality: review scores cluster at 1 and 5 across all price segments — the distribution is not normal, making Spearman rank correlation the more reliable measure.
- **Actionable:** Reducing delivery time will have a greater ROI impact on customer satisfaction than price cuts or freight subsidies.

---
## 7 · Geographic Analysis

**Business question:** Which Brazilian states are the strongest markets? Where are the logistics gaps?

**Why this chart:** Geographic data has spatial structure that tables and bar charts cannot show — neighbouring states tend to share infrastructure, and the North/South divide is immediately visible on a map. A choropleth with rich hover data lets a stakeholder explore all metrics at once without needing multiple charts.

In [ ]:
geo = (
    df.dropna(subset=['customer_state'])
    .groupby('customer_state', observed=True)
    .agg(
        revenue=('payment_value', 'sum'),
        orders=('order_id', 'nunique'),
        avg_ticket=('payment_value', 'mean'),
        avg_review=('review_score', 'mean'),
        avg_delivery=('delivery_days_actual', 'mean'),
    )
    .reset_index()
)
geo.columns = ['state', 'revenue', 'orders', 'avg_ticket', 'avg_review', 'avg_delivery']
geo = geo.sort_values('revenue', ascending=False)
geo.head()

In [ ]:
# Choropleth map of Brazil
fig_map = px.choropleth(
    geo,
    geojson='https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson',
    locations='state',
    featureidkey='properties.sigla',
    color='revenue',
    color_continuous_scale='Blues',
    hover_name='state',
    hover_data={
        'revenue': ':,.0f',
        'orders': ':,',
        'avg_ticket': ':.2f',
        'avg_review': ':.2f',
        'avg_delivery': ':.1f',
    },
    title='Total Revenue by Brazilian State',
    labels={'revenue': 'Revenue (R$)'},
    scope='south america',
)
fig_map.update_geos(fitbounds='locations', visible=False)
fig_map.update_layout(
    height=600,
    margin=dict(t=50, l=0, r=0, b=0),
    font_family='DejaVu Sans',
    coloraxis_colorbar=dict(title='Revenue (R$)'),
)
fig_map.write_image('../outputs/figures/06_choropleth_brazil.png', scale=2)
fig_map.show()

In [ ]:
# State ranking by review score
geo_review = geo.sort_values('avg_review', ascending=True)

# Region mapping (approximate)
region_map = {
    'AC':'N','AM':'N','AP':'N','PA':'N','RO':'N','RR':'N','TO':'N',
    'AL':'NE','BA':'NE','CE':'NE','MA':'NE','PB':'NE','PE':'NE',
    'PI':'NE','RN':'NE','SE':'NE',
    'DF':'CO','GO':'CO','MS':'CO','MT':'CO',
    'ES':'SE','MG':'SE','RJ':'SE','SP':'SE',
    'PR':'S','RS':'S','SC':'S',
}
region_colors = {'N': PALETTE['danger'], 'NE': PALETTE['warning'],
                 'CO': PALETTE['secondary'], 'SE': PALETTE['primary'],
                 'S': PALETTE['accent']}

fig, ax = plt.subplots(figsize=(10, 9))
bar_colors = [region_colors.get(region_map.get(s, 'SE'), PALETTE['neutral'])
              for s in geo_review['state']]
ax.barh(geo_review['state'], geo_review['avg_review'],
        color=bar_colors, edgecolor='white')
ax.axvline(geo_review['avg_review'].mean(), color='black',
           linestyle='--', linewidth=1, label='National avg')
ax.set_xlabel('Avg Review Score')
ax.set_title('Average Review Score by State\n(colored by region)')

# Legend
legend_handles = [mpatches.Patch(color=c, label=r)
                  for r, c in region_colors.items()]
ax.legend(handles=legend_handles, title='Region', loc='lower right')
plt.tight_layout()
save_fig(fig, '06_review_by_state')
plt.show()

### 💡 Conclusion — Geographic Analysis

- **São Paulo** dominates revenue (>40% of total), but this also means the business is highly concentrated — any SP-specific disruption has outsized impact.
- **Northern states (North and Northeast regions)** show consistently lower review scores, driven primarily by longer delivery times rather than product quality issues.
- **Southern states** (PR, SC, RS) show strong review scores and above-average tickets, making them underexploited high-quality markets.
- **Actionable:** Diversifying fulfillment into SP interior and the Northeast could reduce both concentration risk and delivery-related NPS losses simultaneously.

---
## 8 · Key Insights Summary

### Top 7 Findings for Business Stakeholders

---

**1. The business grew ~135% year-over-year (2017→2018)**
Revenue nearly doubled in a single year, with consistent monthly growth throughout both years. This is not a one-time spike — the underlying trend (3-month moving average) is structurally upward.

---

**2. Q4 (November–December) is the most critical revenue period**
A seasonal peak tied to Black Friday and Christmas accounts for disproportionate revenue. Missing logistics capacity in October risks the single most important selling window of the year.

---

**3. Customers shop during business hours, mostly on weekdays**
The busiest window is Tuesday–Thursday, 11 AM–2 PM. Marketing campaigns, push notifications, and flash sales timed to this window will reach the highest-intent audience.

---

**4. Delivery time is the #1 driver of customer dissatisfaction — not price**
Correlation analysis shows delivery days have the strongest (negative) relationship with review scores. A product can be expensive — customers don't penalise that — but a late delivery reliably produces a 1-star review.

---

**5. Northern states have both the worst delivery times and the lowest satisfaction scores**
This is a compound problem: geography makes delivery harder, which tanks reviews, which may suppress repeat purchases in those regions. Logistics investment in the North has the highest customer satisfaction ROI.

---

**6. Health & Beauty and Electronics are the star categories — but also the most satisfaction-sensitive**
They generate the most revenue and have high ticket sizes. A drop in their review scores (e.g., counterfeit products, delayed shipments) would have an outsized impact on overall platform NPS.

---

**7. The business is dangerously concentrated in São Paulo**
SP accounts for >40% of orders and revenue. Southern states (RS, SC, PR) have high review scores and above-average spend but are underrepresented in volume — they represent the clearest geographic growth opportunity.

---

### Suggested Next Steps

| Priority | Analysis | Business Value |
|:---:|---|---|
| 🔴 High | **Delivery delay prediction model** (regression/classification on order features) | Proactively flag at-risk orders before the customer is impacted |
| 🔴 High | **Customer segmentation (RFM or clustering)** | Identify high-LTV customers to protect and lapsed customers to reactivate |
| 🟡 Medium | **Category-level churn analysis** | Which categories have declining repeat purchase rates? |
| 🟡 Medium | **Seller performance scorecard** | Identify seller-level quality issues before they become review problems |
| 🟢 Low | **Price elasticity by category** | Understand whether price changes affect volume (inform promotion strategy) |